# StrikerAPI: a season of soccer in five minutes

This notebook loads the free StrikerAPI season pack (the last completed season, as CSV), answers a few questions offline, then asks the live API the same kind of question with a free key.

Data: StrikerAPI (strikerapi.com), under the [StrikerAPI Sample Data Licence](https://github.com/sports-data-hq/strikerapi-mcp/blob/main/DATA-LICENSE.md).

In [ ]:
import io, zipfile, urllib.request
import pandas as pd

URL = 'https://github.com/sports-data-hq/strikerapi-mcp/releases/download/season-pack/strikerapi-season-pack.zip'
z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read()))
tables = {n[:-4]: pd.read_csv(z.open(n)) for n in z.namelist() if n.endswith('.csv')}

def t(league, kind):
    """One table, e.g. t('soccer', 'players')."""
    return next(df for name, df in tables.items() if f'-{league}-' in name and name.endswith('-' + kind))

sorted(tables)[:12]

## The final table

In [ ]:
standings = t('soccer', 'standings')
standings[['rank', 'team', 'played', 'points', 'goal_diff']].head(10)

## Who finished above or below their chances (our xG)

In [ ]:
teams = t('soccer', 'teams')
teams['goal_diff'] = teams.goals_for - teams.goals_against
teams['xg_diff'] = teams.xg_for - teams.xg_against
ax = teams.plot.scatter(x='xg_diff', y='goal_diff', figsize=(7, 6), title='Goal difference vs xG difference')
for _, r in teams.iterrows():
    ax.annotate(r.team, (r.xg_diff, r.goal_diff), fontsize=7)
ax.axline((0, 0), slope=1, color='grey', lw=0.8)

## Finishing against expected: goals minus non-penalty xG, minimum 1,000 minutes

In [ ]:
p = t('soccer', 'players')
p = p[p.minutes >= 1000].assign(np_goals=lambda d: d.goals - d.penalty_goals, np_goals_minus_np_xg=lambda d: d.np_goals - d.np_xg)
p.sort_values('np_goals_minus_np_xg', ascending=False)[['player', 'team', 'minutes', 'np_goals', 'np_xg', 'np_goals_minus_np_xg']].head(10)

## The same, live

Every earlier season, play-by-play, lineups, odds and today's games are on the StrikerAPI API. Get a free key and paste it below: one key for every sport at [app.sportsdatahq.com](https://app.sportsdatahq.com/signup?utm_source=colab&utm_medium=notebook&utm_campaign=strikerapi-notebook) (1,000 credits a month), which this notebook uses.

In [ ]:
import getpass, json, urllib.request
KEY = getpass.getpass('API key: ')
req = urllib.request.Request('https://api.sportsdatahq.com/v1/soccer/get_standings', data=b'{"league": "laliga"}',
    headers={'x-api-key': KEY, 'content-type': 'application/json'})
res = urllib.request.urlopen(req)
print('credits used:', res.headers.get('x-credits-used'), '| left:', res.headers.get('x-credits-remaining'))
json.loads(res.read())['data']

Ask the same questions in plain English: add the MCP server `https://mcp.sportsdatahq.com/mcp` to Claude or ChatGPT, or install the soccer skills from https://github.com/sports-data-hq/strikerapi-mcp.